# 03 · ELBO：先在可穷举模型上看见 gap
$\log p(y)=\mathcal L(q)+\mathrm{KL}(q\Vert p(z|y))$。
用两个独立先验的 Bernoulli 隐变量和一个 Poisson 观测，检验 mean-field family 的限制。此处网格优化不是大模型 VI 算法，而是可核对的教学参考。


In [ ]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(root))
import numpy as np
import matplotlib.pyplot as plt
from src.hmm import forward_backward, brute_force, poisson_log_emissions
rng=np.random.default_rng(7)


In [ ]:
from itertools import product
s=np.array(list(product([0,1],repeat=2)))
y=np.array([4]); prior=np.array([.49,.21,.21,.09])
logjoint=np.log(prior)+poisson_log_emissions(y,1+4*s.sum(axis=1))[0]
logZ=np.logaddexp.reduce(logjoint); post=np.exp(logjoint-logZ)
best=(-np.inf,None)
for p in np.linspace(.01,.99,99):
    for q in np.linspace(.01,.99,99):
        w=np.array([(1-p)*(1-q),(1-p)*q,p*(1-q),p*q])
        elbo=np.sum(w*(logjoint-np.log(w)))
        if elbo>best[0]: best=(elbo,w)
elbo,w=best; kl=np.sum(w*(np.log(w)-np.log(post)))
assert np.allclose(logZ,elbo+kl)
print('exact posterior:',post,'mean-field:',w)
print('log evidence, ELBO, gap:',logZ,elbo,kl)


## 从这里到 structured VI
逐时间独立 $q(z)=\prod_{m,t}q(z_{mt})$ 会同时丢失时间与 emitter 依赖。链结构 $q(z)=\prod_m q_m(z_{m,1:T})$ 保留单 emitter 时间依赖，但忽略 emitter 间耦合。每个 $q_m$ 可用前向后向计算；坐标更新需对其他链求期望，而非简单减去平均 photon count。

练习：固定 q 为精确 posterior 验证 gap=0。解释 reverse KL 为什么可能偏好某一个解释。ELBO 更高是否必然意味着参数更接近真值？
